In [30]:
import json
import re
import random
import pandas as pd
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import classification_report


# ---------------------------------------------------------------------------
# 1. Wczytanie danych
# ---------------------------------------------------------------------------

def load_data(filepath: str) -> pd.DataFrame:
    """Wczytuje plik JSON z listą obiektów do DataFrame."""
    with open(filepath, encoding="utf-8") as f:
        records = json.load(f)
    return pd.DataFrame(records)


def preprocess_text(text: str) -> str:
    """
    Zamienia znaki nowej linii na spacje i usuwa nadmiarowe białe znaki.
    Dzięki temu TfidfVectorizer traktuje każdą linijkę jako część jednego dokumentu.
    """
    text = text.replace("\n", " ")
    text = re.sub(r"\s+", " ", text).strip()
    return text


train_df = load_data("train.json")
test_df  = load_data("test.json")

train_df["label"] = train_df["label"].str.strip().replace("Oki", "OKI")
test_df["label"]  = test_df["label"].str.strip().replace("Oki", "OKI")

# Zachowaj oryginalny tekst (z \n) do ładnego wyświetlenia
test_df["text_original"] = test_df["text"]

# Preprocessing kolumny text przed wektoryzacją
train_df["text"] = train_df["text"].apply(preprocess_text)
test_df["text"]  = test_df["text"].apply(preprocess_text)

X_train, y_train = train_df["text"], train_df["label"]
X_test,  y_test  = test_df["text"],  test_df["label"]

print(f"Zbiór treningowy: {len(train_df)} próbek, klasy: {sorted(y_train.unique())}")
print(f"Zbiór testowy:    {len(test_df)} próbek\n")


# ---------------------------------------------------------------------------
# 2. Preprocessing i Pipeline
# ---------------------------------------------------------------------------

POLISH_STOP_WORDS = [
    "i", "w", "z", "do", "na", "się", "że", "to", "jak", "po", "nie",
    "a", "o", "co", "ale", "bo", "już", "jej", "jego", "ich", "go",
    "mu", "mi", "mnie", "nas", "nam", "was", "wam", "im", "ją", "je",
    "ten", "ta", "te", "tego", "tej", "tych", "tym", "te", "temu",
    "przez", "przy", "za", "ze", "czy", "od", "dla", "we", "też",
    "oraz", "więc", "gdy", "kiedy", "jestem", "jest", "być", "ma",
    "mam", "masz", "są", "tylko", "tak", "no", "tu", "tam",
]

pipeline = Pipeline([
    (
        "tfidf",
        TfidfVectorizer(
            ngram_range=(1, 2),
            lowercase=True,
            stop_words=POLISH_STOP_WORDS,
            sublinear_tf=True,
            min_df=2,
        ),
    ),
    (
        "clf",
        MultinomialNB(alpha=0.1),
    ),
])


# ---------------------------------------------------------------------------
# 3. Trening i ewaluacja
# ---------------------------------------------------------------------------

pipeline.fit(X_train, y_train)
y_pred = pipeline.predict(X_test)

print("=" * 60)
print("RAPORT KLASYFIKACJI")
print("=" * 60)
print(classification_report(y_test, y_pred, zero_division=0))


# ---------------------------------------------------------------------------
# 4. Analiza losowego fragmentu (sekcji) ze zbioru testowego
# ---------------------------------------------------------------------------

def analyze_random_section(df: pd.DataFrame, seed: int | None = None) -> None:
    """
    Losuje pojedynczy fragment (sekcję) ze zbioru testowego, przewiduje wykonawcę
    i pokazuje pełny tekst razem z porównaniem do prawdziwej etykiety.
    """
    if seed is not None:
        random.seed(seed)

    idx = random.randrange(len(df))
    row = df.iloc[idx]

    text_for_model = row["text"]
    full_text      = row.get("text_original", row["text"])
    true_label     = row["label"]
    song           = row["song"]
    section        = row["section"]

    pred = pipeline.predict([text_for_model])[0]
    probabilities = pipeline.predict_proba([text_for_model])[0]
    classes = list(pipeline.classes_)
    pred_prob = probabilities[classes.index(pred)] * 100

    print("\n" + "=" * 70)
    print("LOSOWY FRAGMENT ZE ZBIORU TESTOWEGO")
    print("=" * 70)
    print(f"Piosenka : \"{song}\"")
    print(f"Sekcja   : {section}")
    print("-" * 70)
    print("Tekst fragmentu:")
    print(full_text)
    print("-" * 70)
    print(f"Przewidziany wykonawca : {pred}  ({pred_prob:.1f}%)")
    print(f"Prawdziwy wykonawca    : {true_label}")
    print(f"Trafienie              : {'TAK' if pred == true_label else 'NIE'}")

    print("\nTop 5 typów modelu:")
    ranked = sorted(zip(classes, probabilities), key=lambda x: x[1], reverse=True)[:5]
    for artist, prob in ranked:
        bar = "█" * int(prob * 40)
        print(f"  {artist:<20} {prob * 100:5.1f}%  {bar}")


# Uruchom analizę. Podaj seed=<liczba>, żeby wylosować zawsze ten sam fragment.
analyze_random_section(test_df, seed=None)

Zbiór treningowy: 6815 próbek, klasy: ['Avi', 'Bedoes', 'Diho', 'Guzior', 'Kabe', 'Louis Villain', 'Malik Montana', 'Mata', 'OG Olgierd', 'OKI', 'Pezet', 'Quebonafide', 'Sentino', 'Sobel', 'White 2115', 'Young Igi', 'Young Leosia', 'Zeamsone', 'Żabson']
Zbiór testowy:    1203 próbek

RAPORT KLASYFIKACJI
               precision    recall  f1-score   support

          Avi       1.00      0.12      0.21        26
       Bedoes       0.58      0.75      0.65        80
         Diho       0.92      0.25      0.39        44
       Guzior       0.75      0.09      0.17        32
         Kabe       1.00      0.53      0.69        47
Louis Villain       0.00      0.00      0.00         5
Malik Montana       0.77      0.55      0.64        83
         Mata       0.88      0.44      0.58        64
   OG Olgierd       1.00      0.38      0.55         8
          OKI       0.89      0.69      0.77        70
        Pezet       0.46      0.82      0.59        82
  Quebonafide       0.67      0.48